# ResNet50 전이학습 - 녹내장 3-클래스 분류 (검증셋 분리 + 성능 개선)

안저(fundus) 이미지로 **advanced_glaucoma / early_glaucoma / normal_control** 3-클래스 분류.

## 이 노트북의 핵심 (11번 노트북과의 차이)

| 항목 | 11번 노트북 | 이 노트북 (12번) |
|---|---|---|
| 데이터 분리 | train / test 2개 | **train / validation / test 3개** |
| 모델 선택 기준 | `test` 성능 → **낙관 편향** | **`validation` 성능** (test는 마지막 1회만) |
| 불균형 처리 | class_weight | class_weight (동일) |
| 추론 | 단일 예측 | **TTA**(좌우반전 평균) + **임계값 튜닝** |

> **왜 검증셋을 분리하나?**
> 학습 중에 test 점수를 보고 "이 epoch 모델이 제일 좋다"고 고르면, 그 점수는
> test 셋에 우연히 맞은 값이 섞여 부풀려진다(낙관 편향). validation 셋으로 모델을 고르고
> test 셋은 학습이 끝난 뒤 **딱 한 번만** 평가해야 일반화 성능을 정직하게 알 수 있다.

---
> **환경**: GPU RTX 4060 Laptop 8GB 기준 전체 학습 ~10분.
> GPU 난수/부동소수점 비결정성 때문에 실행할 때마다 수치가 몇 %p 흔들릴 수 있다.
> 아래 출력은 한 번의 Run All 결과이며, 결론(§10)은 이 실행 기준으로 정리했다.


## 0. 준비

In [1]:
import os, json, warnings
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
os.environ["TF_FORCE_GPU_ALLOW_GROWTH"] = "true"   # GPU 메모리 점진 할당
warnings.filterwarnings("ignore")

import numpy as np
import tensorflow as tf
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.applications.resnet50 import preprocess_input
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.preprocessing import image
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import Callback, ReduceLROnPlateau, EarlyStopping
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, confusion_matrix, recall_score

tf.random.set_seed(42); np.random.seed(42)

DATA = "image_data/glaucoma"        # glaucoma/train/<class>/*, glaucoma/test/<class>/*
IMG  = (224, 224)
BS   = 32
BEST = "best_glaucoma.keras"        # 학습 중 val 기준 best model 저장 경로

I0000 00:00:1788332531.703473   66820 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1788332532.738683   66820 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


### 데이터 압축풀기 (최초 1회)

In [2]:
import zipfile
if not os.path.isdir(DATA):
    with zipfile.ZipFile("image_data/glaucoma.zip") as f:
        f.extractall("./image_data")
    print("압축 해제 완료")

## 1. 데이터 준비 — train / validation / test 분리

`ImageDataGenerator(validation_split=0.2)` + `subset=` 로 **train 폴더**를 다시
학습용 80% / 검증용 20% 로 나눈다 (클래스별 비율 유지 = 층화).
`test` 폴더는 건드리지 않고 맨 마지막 평가에만 쓴다.

- **train** : 증강 O, 학습에만 사용
- **validation** : 증강 X, 모델 선택 / EarlyStopping / 임계값 튜닝
- **test** : 증강 X, 최종 성능 보고 (1회)

In [3]:
# 학습용: 증강 + preprocess_input + validation_split
train_idg = ImageDataGenerator(preprocessing_function=preprocess_input,
                               validation_split=0.2,
                               rotation_range=15,
                               width_shift_range=0.1, height_shift_range=0.1,
                               zoom_range=0.1, horizontal_flip=True, fill_mode="reflect")
# 검증용: 같은 split, 증강 없음
val_idg = ImageDataGenerator(preprocessing_function=preprocess_input, validation_split=0.2)
# 테스트용: split 없음, 증강 없음
test_idg = ImageDataGenerator(preprocessing_function=preprocess_input)

train_data = train_idg.flow_from_directory(f"{DATA}/train", target_size=IMG, batch_size=BS,
                                           class_mode="sparse", subset="training",
                                           shuffle=True, seed=42)
val_data   = val_idg.flow_from_directory(f"{DATA}/train", target_size=IMG, batch_size=BS,
                                         class_mode="sparse", subset="validation",
                                         shuffle=False, seed=42)
test_data  = test_idg.flow_from_directory(f"{DATA}/test", target_size=IMG, batch_size=BS,
                                          class_mode="sparse", shuffle=False)

class_names = [n for n, i in sorted(train_data.class_indices.items(), key=lambda x: x[1])]
NC = len(class_names)
print("classes:", class_names)

Found 1116 images belonging to 3 classes.
Found 278 images belonging to 3 classes.
Found 150 images belonging to 3 classes.
classes: ['advanced_glaucoma', 'early_glaucoma', 'normal_control']


## 2. 클래스 불균형 보정

`early_glaucoma` 표본이 적어(train 1116장 중 약 19%) 그대로 두면 recall 이 0 에 가까워진다.
`class_weight='balanced'` 로 적은 클래스의 손실 가중치를 키운다.

In [4]:
cw = compute_class_weight("balanced", classes=np.arange(NC), y=train_data.classes)
class_weight = {i: float(w) for i, w in enumerate(cw)}
print("class_weight:", class_weight)

class_weight: {0: 1.1038575667655786, 1: 1.7630331753554502, 2: 0.6549295774647887}


## 3. 모델 구성 — ResNet50 전이학습

- `include_top=False` : ImageNet 분류층 제거, 특징 추출부만 사용
- 새 헤드 : GlobalAveragePooling → Dropout → Dense(256) → Dropout → softmax(3)
- 1단계에서는 `base.trainable = False` 로 사전학습 가중치를 동결

In [5]:
base = ResNet50(weights="imagenet", include_top=False, input_shape=(*IMG, 3))

x = GlobalAveragePooling2D()(base.output)
x = Dropout(0.3)(x)
x = Dense(256, activation="relu")(x)
x = Dropout(0.3)(x)
out = Dense(NC, activation="softmax")(x)

model = Model(base.input, out)
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_pad           │ (None, 230, 230,  │          0 │ input_layer[0][0] │
│ (ZeroPadding2D)     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_conv (Conv2D) │ (None, 112, 112,  │      9,472 │ conv1_pad[0][0]   │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_bn            │ (None, 112, 112,  │        256 │ conv1_conv[0][0]  │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_relu          │ (None, 112, 112,  │          0 │ conv1_bn[0][0]    │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pool1_pad           │ (None, 114, 114,  │          0 │ conv1_relu[0][0]  │
│ (ZeroPadding2D)     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pool1_pool          │ (None, 56, 56,    │          0 │ pool1_pad[0][0]   │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_1_conv │ (None, 56, 56,    │      4,160 │ pool1_pool[0][0]  │
│ (Conv2D)            │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_1_bn   │ (None, 56, 56,    │        256 │ conv2_block1_1_c… │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_1_relu │ (None, 56, 56,    │          0 │ conv2_block1_1_b… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_2_conv │ (None, 56, 56,    │     36,928 │ conv2_block1_1_r… │
│ (Conv2D)            │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_2_bn   │ (None, 56, 56,    │        256 │ conv2_block1_2_c… │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_2_relu │ (None, 56, 56,    │          0 │ conv2_block1_2_b… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_0_conv │ (None, 56, 56,    │     16,640 │ pool1_pool[0][0]  │
│ (Conv2D)            │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_3_conv │ (None, 56, 56,    │     16,640 │ conv2_block1_2_r… │
│ (Conv2D)            │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_0_bn   │ (None, 56, 56,    │      1,024 │ conv2_block1_0_c… │
│ (BatchNormalizatio… │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_3_bn   │ (None, 56, 56,    │      1,024 │ conv2_block1_3_c

 Total params: 24,113,027 (91.98 MB)

 Trainable params: 24,059,907 (91.78 MB)

 Non-trainable params: 53,120 (207.50 KB)

## 4. 평가 유틸 — TTA · 임계값 튜닝 · best model 콜백

- **TTA (Test-Time Augmentation)** : 원본 + 좌우반전 예측을 평균 → 예측 안정화
- **임계값 튜닝** : validation 에서 클래스별 나눗셈 계수를 좌표상승법으로 탐색,
  목표 = `macro_recall + accuracy` (모델 선택 기준과 동일해 한 클래스만 희생되는 붕괴를 막음)
- **Track 콜백** : 매 epoch validation 의 `macro_recall + accuracy` 가 최고면 best model 저장

In [6]:
def to_memory(gen):
    X, Y = [], []
    for i in range(len(gen)):
        xb, yb = gen[i]; X.append(xb); Y.append(yb)
    return np.concatenate(X), np.concatenate(Y).astype(int)

Xval, yval = to_memory(val_data)     # 콜백/튜닝에서 반복 사용하므로 메모리에 적재
Xte,  yte  = to_memory(test_data)

def tta_predict(model, X):
    p = model.predict(X, verbose=0) + model.predict(X[:, :, ::-1, :], verbose=0)
    return p / 2

def _score(y, pred):
    return recall_score(y, pred, average="macro", zero_division=0) + (pred == y).mean()

def tune_thresholds(probs, y):
    t = np.ones(NC); best = _score(y, probs.argmax(1))
    for _ in range(20):
        moved = False
        for c in range(NC):
            for cand in np.linspace(0.6, 1.6, 41):
                tt = t.copy(); tt[c] = cand
                s = _score(y, (probs / tt).argmax(1))
                if s > best + 1e-6:
                    best, t, moved = s, tt, True
        if not moved:
            break
    return t

class Track(Callback):
    def __init__(self):
        super().__init__(); self.best = -1.0; self.info = None
    def on_epoch_end(self, epoch, logs=None):
        pred = tta_predict(self.model, Xval).argmax(1)
        acc  = float((pred == yval).mean())
        mrec = float(recall_score(yval, pred, average="macro", zero_division=0))
        tag = ""
        if mrec + acc > self.best:
            self.best = mrec + acc
            self.info = dict(epoch=epoch + 1, val_acc=acc, val_macro_recall=mrec)
            self.model.save(BEST); tag = "  *saved*"
        print(f"  ep{epoch+1:02d}  val_acc={acc:.4f}  val_macro_recall={mrec:.4f}{tag}")

track = Track()

## 5. Phase 1 — 분류 헤드만 학습 (base 동결)

- optimizer : Adam(1e-3), loss : sparse_categorical_crossentropy, metric : accuracy
- `validation_data=val_data` ← **test 아님**

In [7]:
model.compile(optimizer=Adam(1e-3),
              loss="sparse_categorical_crossentropy", metrics=["accuracy"])

model.fit(train_data, validation_data=val_data, epochs=12,
          class_weight=class_weight,
          callbacks=[track,
                     ReduceLROnPlateau(monitor="val_accuracy", mode="max",
                                       patience=4, factor=0.5, min_lr=1e-5)],
          verbose=2)

Epoch 1/12
  ep01  val_acc=0.5108  val_macro_recall=0.3333  *saved*
35/35 - 104s - 3s/step - accuracy: 0.6568 - loss: 0.8633 - val_accuracy: 0.5108 - val_loss: 12122.8555 - learning_rate: 0.0010
Epoch 2/12
  ep02  val_acc=0.5108  val_macro_recall=0.3333
35/35 - 10s - 276ms/step - accuracy: 0.7025 - loss: 0.6854 - val_accuracy: 0.5108 - val_loss: 10069.9600 - learning_rate: 0.0010
Epoch 3/12
  ep03  val_acc=0.5108  val_macro_recall=0.3333
35/35 - 8s - 230ms/step - accuracy: 0.6523 - loss: 0.8845 - val_accuracy: 0.5108 - val_loss: 62974.3164 - learning_rate: 0.0010
Epoch 4/12
  ep04  val_acc=0.5108  val_macro_recall=0.3333
35/35 - 9s - 263ms/step - accuracy: 0.6837 - loss: 0.7265 - val_accuracy: 0.5108 - val_loss: 165.0389 - learning_rate: 0.0010
Epoch 5/12
  ep05  val_acc=0.6511  val_macro_recall=0.5019  *saved*
35/35 - 11s - 316ms/step - accuracy: 0.6953 - loss: 0.6499 - val_accuracy: 0.6511 - val_loss: 12.9249 - learning_rate: 0.0010
Epoch 6/12
  ep06  val_acc=0.6439  val_macro_recall

## 6. Phase 2 — conv4 + conv5 블록 미세조정

상위 블록만 동결 해제하고 아주 작은 학습률(1e-5)로 미세조정.
BatchNormalization 층은 계속 동결(작은 배치에서 통계가 흔들리는 것 방지).

In [8]:
base.trainable = True
for layer in base.layers:
    if not (layer.name.startswith("conv4") or layer.name.startswith("conv5")):
        layer.trainable = False
    if isinstance(layer, tf.keras.layers.BatchNormalization):
        layer.trainable = False

model.compile(optimizer=Adam(1e-5),
              loss="sparse_categorical_crossentropy", metrics=["accuracy"])

model.fit(train_data, validation_data=val_data, epochs=20,
          class_weight=class_weight,
          callbacks=[track,
                     EarlyStopping(monitor="val_accuracy", mode="max", patience=10),
                     ReduceLROnPlateau(monitor="val_accuracy", mode="max",
                                       patience=5, factor=0.5, min_lr=1e-7)],
          verbose=2)

Epoch 1/20
  ep01  val_acc=0.8237  val_macro_recall=0.7867
35/35 - 42s - 1s/step - accuracy: 0.7876 - loss: 0.5337 - val_accuracy: 0.8273 - val_loss: 0.5451 - learning_rate: 1.0000e-05
Epoch 2/20
  ep02  val_acc=0.8381  val_macro_recall=0.8075  *saved*
35/35 - 11s - 305ms/step - accuracy: 0.8109 - loss: 0.4973 - val_accuracy: 0.8273 - val_loss: 0.5104 - learning_rate: 1.0000e-05
Epoch 3/20
  ep03  val_acc=0.8309  val_macro_recall=0.8012
35/35 - 9s - 271ms/step - accuracy: 0.7894 - loss: 0.4949 - val_accuracy: 0.8345 - val_loss: 0.4912 - learning_rate: 1.0000e-05
Epoch 4/20
  ep04  val_acc=0.8345  val_macro_recall=0.8067
35/35 - 8s - 237ms/step - accuracy: 0.7867 - loss: 0.4952 - val_accuracy: 0.8381 - val_loss: 0.4847 - learning_rate: 1.0000e-05
Epoch 5/20
  ep05  val_acc=0.8273  val_macro_recall=0.8004
35/35 - 10s - 291ms/step - accuracy: 0.7921 - loss: 0.4661 - val_accuracy: 0.8309 - val_loss: 0.4903 - learning_rate: 1.0000e-05
Epoch 6/20
  ep06  val_acc=0.8309  val_macro_recall=0.80

## 7. best model 로드 + 임계값 튜닝 (validation 기준)

임계값은 **validation 으로만** 결정한다. test 는 아직 보지 않는다.

In [9]:
print("best (val 기준):", json.dumps(track.info, indent=2))

model = tf.keras.models.load_model(BEST)

thr = tune_thresholds(tta_predict(model, Xval), yval)
print("tuned thresholds:", np.round(thr, 3).tolist())
np.save("glaucoma_thresholds.npy", thr)

best (val 기준): {
  "epoch": 15,
  "val_acc": 0.8453237410071942,
  "val_macro_recall": 0.8194680905948513
}
tuned thresholds: [1.075, 1.0, 1.0]


## 8. 최종 테스트 평가 (딱 1회)

`argmax + TTA` 와 `argmax + TTA + 임계값 튜닝` 두 가지를 비교한다.
여기서 나온 숫자가 **이 모델의 정직한 일반화 성능**이다.
행 = 실제, 열 = 예측 (순서: advanced / early / normal).

In [10]:
proba_test = tta_predict(model, Xte)

for name, pred in [("argmax + TTA",                   proba_test.argmax(1)),
                   ("argmax + TTA + tuned threshold", (proba_test / thr).argmax(1))]:
    print(f"===== TEST : {name} =====")
    print(confusion_matrix(yte, pred))
    print(classification_report(yte, pred, target_names=class_names, digits=4))
    print()

===== TEST : argmax + TTA =====
[[42  3  1]
 [ 7 17  2]
 [ 8 21 49]]
                   precision    recall  f1-score   support

advanced_glaucoma     0.7368    0.9130    0.8155        46
   early_glaucoma     0.4146    0.6538    0.5075        26
   normal_control     0.9423    0.6282    0.7538        78

         accuracy                         0.7200       150
        macro avg     0.6979    0.7317    0.6923       150
     weighted avg     0.7878    0.7200    0.7301       150


===== TEST : argmax + TTA + tuned threshold =====
[[42  3  1]
 [ 6 18  2]
 [ 8 21 49]]
                   precision    recall  f1-score   support

advanced_glaucoma     0.7500    0.9130    0.8235        46
   early_glaucoma     0.4286    0.6923    0.5294        26
   normal_control     0.9423    0.6282    0.7538        78

         accuracy                         0.7267       150
        macro avg     0.7070    0.7445    0.7023       150
     weighted avg     0.7943    0.7267    0.7363       150




## 9. 단일 이미지 예측 (test.png)

`test.png` 는 early_glaucoma (인덱스 1) 로 알려져 있다.

In [12]:
img = image.load_img("image_data/test.png", target_size=IMG)
x = image.img_to_array(img)
x = preprocess_input(np.expand_dims(x, axis=0))   # 학습과 동일 전처리

p = tta_predict(model, x)
print("확률:", np.round(p[0], 4))
print("예측 (argmax)     :", class_names[p[0].argmax()])
print("예측 (임계값 적용) :", class_names[(p[0] / thr).argmax()])

확률: [0.3905 0.5845 0.025 ]
예측 (argmax)     : early_glaucoma
예측 (임계값 적용) : early_glaucoma


## 10. 결과 요약

아래는 위 출력 셀(5~9)에 나온 **실제 실행 결과** 기준 정리다.

### 10-1. 학습 경과 (셀 5·6)

| 단계 | 관찰 |
|---|---|
| Phase 1 초반 | ep1~4 불안정 — val_accuracy 0.5108 고정, val_loss 가 수천~수만으로 폭증 |
| Phase 1 회복 후 | ep7 부터 정상화(val_acc≈0.81), ep12 val_acc 0.831 / val_macro_recall 0.787 |
| Phase 2 (미세조정) | 완만하게 개선, **ep15 에서 best (val_acc 0.845 / val_macro_recall 0.819)** |
| 과적합 | Phase 2 내내 train acc≈0.80, val_loss 0.49→0.52 로 큰 악화는 없음 (미세조정 lr 1e-5 로 억제됨) |

> Phase 1 초반 발산은 헤드에 `Adam(1e-3)` 이 다소 높아서다. 재현 시 **lr=5e-4** 또는
> warmup 을 주면 더 안정적이다. `ReduceLROnPlateau` 덕에 결국 회복은 됐다.

### 10-2. validation vs test  (best model = Phase 2 epoch 15)

| 지표 | validation<br>(모델·임계값 선택) | **test<br>(정직한 성능, 1회)** |
|---|---|---|
| accuracy | 0.845 | **0.720** |
| macro recall | 0.819 | **0.732** |

**val→test 격차 약 9~12%p.** validation(278장)도 작고 best epoch 를 여러 후보에서 골랐으니
약간 낙관적이며, test 폴더의 분포 차이(촬영기기·기관) 가능성도 있다. **test 0.72 가 실제 실력.**

### 10-3. test 클래스별 성능 — argmax + TTA (셀 8)

| 클래스 | precision | recall | f1 | support |
|---|---|---|---|---|
| advanced_glaucoma | 0.737 | **0.913** | 0.816 | 46 |
| early_glaucoma | **0.415** | 0.654 | 0.508 | 26 |
| normal_control | 0.942 | **0.628** | 0.754 | 78 |
| **accuracy** | | | **0.720** | 150 |
| macro avg | 0.698 | 0.732 | 0.692 | 150 |

혼동행렬 (행 = 실제, 열 = 예측):

```
              advanced  early  normal
advanced   [   42        3      1   ]   recall 0.913
early      [    7       17      2   ]   recall 0.654
normal     [    8       21     49   ]   recall 0.628
```

- **glaucoma 쪽으로 치우쳐 예측** — advanced recall 0.91 로 매우 높지만, normal 78장 중 29장을
  glaucoma(early 21 + advanced 8)로 오분류해 **normal recall 0.63** 로 낮다.
- **가장 큰 오차: normal 21장을 early 로 오분류** → early precision 0.42 로 끌어내림.
  경계가 모호한 **early ↔ normal 구분이 핵심 난제**.
- 의료 스크리닝 관점에선 "정상을 의심 환자로 넘기는" 방향이라 안전한 쪽 실수이긴 하다
  (glaucoma 민감도 = advanced+early 를 놓치지 않는 비율은 높음).

### 10-4. 임계값 튜닝 효과 (셀 8, 두 번째 블록)

튜닝 결과 `thresholds = [1.075, 1.0, 1.0]` (advanced 확률만 소폭 낮춤)

| | accuracy | macro recall | early recall |
|---|---|---|---|
| argmax + TTA | 0.720 | 0.732 | 0.654 |
| + tuned threshold | **0.727** | **0.745** | **0.692** |

→ 이번 실행에선 **미세하게 이득** (early 로 애매하던 1건이 advanced→early 로 교정, accuracy·recall 소폭↑).
다만 val(278장)에 맞춘 값이라 이득 폭이 작고 실행마다 흔들릴 수 있다.

### 10-5. test.png 단일 예측 (셀 9)

확률 `[0.39, 0.58, 0.03]` → **early_glaucoma** (정답). 다만 advanced 확률도 0.39 로 꽤 높아 **확신은 낮다**.

### 10-6. 11번 노트북과의 비교

| | 11번<br>(test 로 best epoch 선택) | **12번<br>(val 로 선택, test 1회 평가)** |
|---|---|---|
| 보고 accuracy | 0.807 | **0.720** |
| 보고 macro recall | 0.796 | **0.732** |
| early_glaucoma recall | 0.692 | 0.654 |

11번의 0.81 은 **test 를 보며 가장 잘 맞는 epoch 를 골라 부풀려진 값**이고,
12번의 **0.72 가 정직한 실력**이다. 12번 모델이 더 나쁜 게 아니라, 11번이 채점지를 보고 공부한 것이다.
**성능을 올린 게 아니라 믿을 수 있게 측정**한 것이 이 노트북의 목적. (참고: 12번은 macro recall 은 오히려 더 높다.)

### 10-7. 더 높이려면 (별도 작업)

- **Stratified K-Fold** — fold별 성능 평균±표준편차 + fold 모델 앙상블로 신뢰구간 확보
- Phase 1 lr 낮추기(5e-4) / warmup — 초반 발산 제거
- early ↔ normal 혼동 직접 공략: **2단계 분류**(normal vs glaucoma → early vs advanced) 또는 순서형(ordinal) 모델
- 안저 전용 전처리(망막 크롭 + CLAHE), 입력 해상도 상향, 백본 교체
  (단, 앞선 실험에서 Ben Graham + EfficientNetV2-S 는 이 데이터에서 오히려 낮았음)
- 외부 데이터(REFUGE, ORIGA, RIM-ONE)로 early_glaucoma 표본 확대
